In [3]:
from collections import defaultdict, Counter
import os
import json
import math

all_seed_results = []
all_categories = set()

# =========================
# 各seedの結果を読み込む
# =========================
for seed in range(4):
    path = os.path.join(
        "..", "data", "cossim_bw_categories",
        f"category_similarity_whitened_12b_target_concepts_mini_13_catnum_plus_20_seed{seed}.json"
    )
    with open(path, "r") as f:
        data = json.load(f)

    all_seed_results.append({
        "seed": seed,
        "similarity_matrix": data["similarity_matrix"],
        "classification": data["classification"],
    })

    all_categories.update(data["classification"].keys())

all_categories = sorted(all_categories)

# =========================
# 集計用データ構造
# =========================
# own_cat -> near/far に出たカテゴリの出現回数
near_counter_per_cat = defaultdict(Counter)
far_counter_per_cat = defaultdict(Counter)

# own_cat -> other_cat -> 類似度リスト
sim_values_per_pair = defaultdict(lambda: defaultdict(list))

# own_cat -> seedごとの near/far
near_list_per_cat = defaultdict(list)
far_list_per_cat = defaultdict(list)

# =========================
# 集計
# =========================
for result in all_seed_results:
    seed = result["seed"]
    similarity_matrix = result["similarity_matrix"]
    classification = result["classification"]

    for own_cat, groups in classification.items():
        # near
        near_items = groups.get("near", [])
        for other_cat, score in near_items:
            near_counter_per_cat[own_cat][other_cat] += 1
            near_list_per_cat[own_cat].append({
                "seed": seed,
                "category": other_cat,
                "score": score,
            })

        # far
        far_items = groups.get("far", [])
        for other_cat, score in far_items:
            far_counter_per_cat[own_cat][other_cat] += 1
            far_list_per_cat[own_cat].append({
                "seed": seed,
                "category": other_cat,
                "score": score,
            })

        # similarity 全体
        if own_cat in similarity_matrix:
            for other_cat, score in similarity_matrix[own_cat].items():
                if other_cat == own_cat:
                    continue
                sim_values_per_pair[own_cat][other_cat].append(score)

# =========================
# 統計量計算
# =========================
def calc_mean(xs):
    return sum(xs) / len(xs) if xs else None

def calc_std(xs):
    if len(xs) <= 1:
        return 0.0
    m = calc_mean(xs)
    return math.sqrt(sum((x - m) ** 2 for x in xs) / len(xs))

summary_per_cat = {}

for own_cat in all_categories:
    near_summary = []
    far_summary = []
    sim_summary = []

    # そのカテゴリに対する全候補カテゴリ
    candidate_cats = set(sim_values_per_pair[own_cat].keys()) \
        | set(near_counter_per_cat[own_cat].keys()) \
        | set(far_counter_per_cat[own_cat].keys())

    for other_cat in sorted(candidate_cats):
        sim_list = sim_values_per_pair[own_cat][other_cat]
        near_count = near_counter_per_cat[own_cat][other_cat]
        far_count = far_counter_per_cat[own_cat][other_cat]

        item = {
            "category": other_cat,
            "mean_similarity": calc_mean(sim_list),
            "std_similarity": calc_std(sim_list),
            "num_observed": len(sim_list),
            "near_count": near_count,
            "far_count": far_count,
        }
        sim_summary.append(item)

        if near_count > 0:
            near_summary.append(item)
        if far_count > 0:
            far_summary.append(item)

    # near は near_count 多い順、同率なら mean_similarity 高い順
    near_summary = sorted(
        near_summary,
        key=lambda x: (-x["near_count"], -(x["mean_similarity"] if x["mean_similarity"] is not None else -999))
    )

    # far は far_count 多い順、同率なら mean_similarity 低い順
    far_summary = sorted(
        far_summary,
        key=lambda x: (-x["far_count"], (x["mean_similarity"] if x["mean_similarity"] is not None else 999))
    )

    # 全ペアの平均類似度ランキング
    sim_high_to_low = sorted(
        sim_summary,
        key=lambda x: -(x["mean_similarity"] if x["mean_similarity"] is not None else -999)
    )
    sim_low_to_high = sorted(
        sim_summary,
        key=lambda x: (x["mean_similarity"] if x["mean_similarity"] is not None else 999)
    )

    summary_per_cat[own_cat] = {
        "near_candidates_ranked": near_summary,
        "far_candidates_ranked": far_summary,
        "most_similar_by_mean": sim_high_to_low[:10],
        "least_similar_by_mean": sim_low_to_high[:10],
        "near_by_seed": near_list_per_cat[own_cat],
        "far_by_seed": far_list_per_cat[own_cat],
    }

# =========================
# 表示
# =========================
for own_cat in all_categories:
    print("=" * 80)
    print(f"OWN CATEGORY: {own_cat}")

    print("\n[Near に現れたカテゴリランキング]")
    for item in summary_per_cat[own_cat]["near_candidates_ranked"][:10]:
        print(
            f"  {item['category']:<35} "
            f"near_count={item['near_count']}  "
            f"mean_sim={item['mean_similarity']:.4f}  "
            f"std={item['std_similarity']:.4f}  "
            f"far_count={item['far_count']}"
        )

    print("\n[Far に現れたカテゴリランキング]")
    for item in summary_per_cat[own_cat]["far_candidates_ranked"][:10]:
        print(
            f"  {item['category']:<35} "
            f"far_count={item['far_count']}  "
            f"mean_sim={item['mean_similarity']:.4f}  "
            f"std={item['std_similarity']:.4f}  "
            f"near_count={item['near_count']}"
        )

    print("\n[平均類似度が高いカテゴリ TOP10]")
    for item in summary_per_cat[own_cat]["most_similar_by_mean"]:
        print(
            f"  {item['category']:<35} "
            f"mean_sim={item['mean_similarity']:.4f}  "
            f"std={item['std_similarity']:.4f}  "
            f"near_count={item['near_count']}  "
            f"far_count={item['far_count']}"
        )

    print("\n[平均類似度が低いカテゴリ TOP10]")
    for item in summary_per_cat[own_cat]["least_similar_by_mean"]:
        print(
            f"  {item['category']:<35} "
            f"mean_sim={item['mean_similarity']:.4f}  "
            f"std={item['std_similarity']:.4f}  "
            f"near_count={item['near_count']}  "
            f"far_count={item['far_count']}"
        )

# =========================
# JSON保存
# =========================
output_path = os.path.join(
    "..", "data", "cossim_bw_categories",
    "aggregated_near_far_analysis_across_seeds.json"
)

with open(output_path, "w") as f:
    json.dump(summary_per_cat, f, indent=4)

print("\nSaved aggregated result to:", output_path)

OWN CATEGORY: Algorithm

[Near に現れたカテゴリランキング]
  activity                            near_count=3  mean_sim=0.5543  std=0.0962  far_count=0
  card game                           near_count=1  mean_sim=0.4576  std=0.0527  far_count=0

[Far に現れたカテゴリランキング]
  castle                              far_count=2  mean_sim=-0.4119  std=0.0091  near_count=0
  monument                            far_count=2  mean_sim=-0.3864  std=0.0486  near_count=0

[平均類似度が高いカテゴリ TOP10]
  activity                            mean_sim=0.5543  std=0.0962  near_count=3  far_count=0
  card game                           mean_sim=0.4576  std=0.0527  near_count=1  far_count=0
  Medicine                            mean_sim=0.4322  std=0.0494  near_count=0  far_count=0
  anatomical structure                mean_sim=0.4173  std=0.0166  near_count=0  far_count=0
  sport                               mean_sim=0.3800  std=0.0191  near_count=0  far_count=0
  protein                             mean_sim=0.3782  std=0.0499  near_

In [7]:
summary_per_cat['Algorithm']['least_similar_by_mean'][0]['category']

'castle'